# DREAM AI — train our native model from random initialization

This notebook trains DREAM's own PyTorch transformer. It does **not** download or call another model, use a third-party AI API, or import pretrained weights. The checkpoint starts with random weights.

**Important:** the current architecture is an early research-scale model, not a capable general assistant. Training requires a substantial, high-quality dataset that you have permission to use. Do not mark rights/privacy/quality review as complete unless a person has actually completed those checks. Free GPU availability and session duration are not guaranteed.

In [ ]:
sft_completed = False
from pathlib import Path
import os, subprocess, sys, torch

REPO = Path('/content/dream-ai')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/mjnr0477/dream-ai.git', str(REPO)], check=True)
os.chdir(REPO)

# Set True to keep checkpoints in Google Drive across Colab disconnects.
# Drive will ask you to authorize access. False keeps files only in this session.
USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_DIR = Path('/content/drive/MyDrive/DREAM AI/checkpoints')
else:
    CHECKPOINT_DIR = Path('checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
PRETRAIN_CHECKPOINT = CHECKPOINT_DIR / 'dream-small.pt'
SFT_CHECKPOINT = CHECKPOINT_DIR / 'dream-instruct.pt'

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. You can run a tiny smoke test on CPU, but substantial training will be slow.')

## 1. Upload the reviewed DREAM corpus

Prepare a JSONL file named `approved-records.jsonl`. Each line must contain `id`, `source`, `source_group`, `license`, `source_date` (`YYYY-MM-DD`), `rights_confirmed`, `privacy_reviewed`, `quality_reviewed`, `language`, and `text`. The three review fields must be `true` only after real human review. Use at least three independent source groups. Keep personal or confidential information out of the corpus. The generated manifest summarizes language and license counts and the date range for an audit before training.

In [ ]:
from google.colab import files
uploaded = files.upload()
if 'approved-records.jsonl' not in uploaded:
    raise RuntimeError('Please upload a file named approved-records.jsonl')
Path('data').mkdir(exist_ok=True)
Path('data/approved-records.jsonl').write_bytes(uploaded['approved-records.jsonl'])
print('Uploaded bytes:', Path('data/approved-records.jsonl').stat().st_size)

In [ ]:
subprocess.run([sys.executable, 'model/prepare_corpus.py', '--input', 'data/approved-records.jsonl', '--output', 'data/prepared'], check=True)
import json
manifest = json.loads(Path('data/prepared/manifest.json').read_text())
print(json.dumps(manifest, indent=2))
for split in ('train', 'validation', 'test'):
    size = Path(f'data/prepared/{split}.txt').stat().st_size
    print(split, 'UTF-8 bytes:', size)
    if size < 258:
        raise RuntimeError(f'{split} split is too small for the default 256-byte context. Add more reviewed data before training.')

## 2. Train DREAM from scratch

The starter settings use a small micro-batch plus gradient accumulation to reduce peak memory. Increase steps only if the runtime can finish. Checkpoints are saved atomically. If you enable Google Drive above, they survive Colab session resets; otherwise they exist only until this runtime ends. Re-running this cell resumes from the existing DREAM checkpoint instead of restarting from random weights.

In [ ]:
TRAIN_STEPS = 2000
# Used only for a brand-new checkpoint. Resume restores the saved tokenizer.
TOKENIZER_FOR_NEW_RUN = 'bpe'  # choose 'byte' to retain the original byte tokenizer
BPE_VOCAB_SIZE = 2048
MICRO_BATCH = 4
ACCUMULATION = 8
train_command = [
    sys.executable, 'model/train.py',
    '--data', 'data/prepared/train.txt',
    '--validation-data', 'data/prepared/validation.txt',
    '--manifest', 'data/prepared/manifest.json',
    '--out', str(PRETRAIN_CHECKPOINT),
    '--steps', str(TRAIN_STEPS),
    '--batch-size', str(MICRO_BATCH),
    '--grad-accum-steps', str(ACCUMULATION),
    '--block-size', '256', '--d-model', '256', '--layers', '6', '--heads', '8',
    '--eval-every', '100', '--checkpoint-every', '250', '--eval-batches', '10'
]
if PRETRAIN_CHECKPOINT.exists():
    train_command += ['--resume', str(PRETRAIN_CHECKPOINT)]
    print('Resuming existing DREAM pretraining checkpoint and restoring its tokenizer.')
else:
    train_command += ['--tokenizer', TOKENIZER_FOR_NEW_RUN]
    if TOKENIZER_FOR_NEW_RUN == 'bpe':
        train_command += ['--tokenizer-vocab-size', str(BPE_VOCAB_SIZE)]
subprocess.run(train_command, check=True)

## 3. Evaluate on the held-out test split

This reports next-token prediction metrics and a sample generation. These metrics do not prove reasoning, factual accuracy, safety or instruction-following ability.

In [ ]:
subprocess.run([
    sys.executable, 'model/evaluate.py',
    '--checkpoint', str(PRETRAIN_CHECKPOINT),
    '--data', 'data/prepared/test.txt',
    '--manifest', 'data/prepared/manifest.json',
    '--batches', '50', '--batch-size', '8',
    '--out', 'eval-results/dream-small.json'
], check=True)
report = json.loads(Path('eval-results/dream-small.json').read_text())
print('Tokenizer:', report['tokenizer'])
print('Bits per token:', report['bits_per_token'])
print('Perplexity per token:', report['perplexity_per_token'])
print('Sample:', report['sample_generation'])

## 4. Optional: instruction-tune DREAM on reviewed conversations

This second stage starts from the DREAM checkpoint you just trained. It does not load another model. Provide separate `sft-train.jsonl` and `sft-validation.jsonl` files with rights/privacy/quality review completed. The script masks user/system prompts and learns only from assistant responses. Their source groups must be disjoint from each other and from the held-out pretraining test sources in `data/prepared/manifest.json`.

In [ ]:
uploaded_sft = files.upload()
required_sft = ('sft-train.jsonl', 'sft-validation.jsonl')
sft_available = all(name in uploaded_sft for name in required_sft)
if sft_available:
    Path('data').mkdir(exist_ok=True)
    for name in required_sft:
        Path('data', name).write_bytes(uploaded_sft[name])
    print('Training bytes:', Path('data/sft-train.jsonl').stat().st_size)
    print('Validation bytes:', Path('data/sft-validation.jsonl').stat().st_size)
else:
    print('No complete SFT dataset uploaded. Skipping instruction tuning; pretraining remains available.')

In [ ]:
if sft_available:
    SFT_STEPS = 1000
    sft_command = [
        sys.executable, 'model/sft.py',
        '--checkpoint', str(PRETRAIN_CHECKPOINT),
        '--train-data', 'data/sft-train.jsonl',
        '--validation-data', 'data/sft-validation.jsonl',
        '--pretraining-manifest', 'data/prepared/manifest.json',
        '--pretraining-test-data', 'data/prepared/test.txt',
        '--out', str(SFT_CHECKPOINT),
        '--steps', str(SFT_STEPS),
        '--batch-size', '2', '--grad-accum-steps', '4',
        '--eval-every', '100', '--checkpoint-every', '250',
        '--eval-batches', '10'
    ]
    if SFT_CHECKPOINT.exists():
        sft_command += ['--resume', str(SFT_CHECKPOINT)]
        print('Resuming existing DREAM instruction-tuning checkpoint.')
    subprocess.run(sft_command, check=True)
    sft_completed = True
else:
    print('Instruction tuning skipped. The native pretraining checkpoint is still available.')

In [ ]:
if sft_completed:
    subprocess.run([
        sys.executable, 'model/evaluate.py',
        '--checkpoint', str(SFT_CHECKPOINT),
        '--data', 'data/prepared/test.txt',
    '--manifest', 'data/prepared/manifest.json',
        '--batches', '50', '--batch-size', '8',
        '--out', 'eval-results/dream-instruct.json'
    ], check=True)
    sft_report = json.loads(Path('eval-results/dream-instruct.json').read_text())
    print('Instruction checkpoint bits per token:', sft_report['bits_per_token'])
    print('Sample:', sft_report['sample_generation'])
else:
    print('No instruction checkpoint to evaluate; pretraining evaluation above is complete.')

## 5. Generate DREAM's native task benchmark answers

This runs the checkpoint trained above against the 40-case DREAM task set. The output contains each prompt, rubric and native response for independent human scoring. No external model grades the answers; a successful run is not a quality certification.


In [ ]:
BENCHMARK_CHECKPOINT = SFT_CHECKPOINT if sft_completed else PRETRAIN_CHECKPOINT
TASK_BENCHMARK_REPORT = Path('eval-results/dream-native-human-review.json')
subprocess.run([
    sys.executable, 'model/run_task_benchmark.py',
    '--checkpoint', str(BENCHMARK_CHECKPOINT),
    '--benchmark', 'evals/core-v1.jsonl',
    '--out', str(TASK_BENCHMARK_REPORT),
    '--max-new-tokens', '128', '--temperature', '0'
], check=True)
task_report = json.loads(TASK_BENCHMARK_REPORT.read_text(encoding='utf-8'))
print('Native benchmark tasks:', task_report['task_count'])
print('Checkpoint SHA-256:', task_report['checkpoint_sha256'])
print('Human scoring required for every response.')


## 6. Package the reproducible DREAM training artifacts

The bundle contains the checkpoints, evaluation reports, exact pretraining train/validation/test text splits and their manifest. If SFT completed, it also includes the reviewed SFT train/validation JSONL files and instruction checkpoint. Keep this bundle private if your reviewed data should not be shared.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from hashlib import sha256
artifact_entries = [
    (PRETRAIN_CHECKPOINT, 'checkpoints/dream-small.pt'),
    (Path('eval-results/dream-small.json'), 'eval-results/dream-small.json'),
    (TASK_BENCHMARK_REPORT, 'eval-results/dream-native-human-review.json'),
    (Path('data/prepared/manifest.json'), 'data/prepared/manifest.json'),
    (Path('data/prepared/train.txt'), 'data/prepared/train.txt'),
    (Path('data/prepared/validation.txt'), 'data/prepared/validation.txt'),
    (Path('data/prepared/test.txt'), 'data/prepared/test.txt')
]
if sft_completed:
    artifact_entries += [
        (SFT_CHECKPOINT, 'checkpoints/dream-instruct.pt'),
        (Path('eval-results/dream-instruct.json'), 'eval-results/dream-instruct.json'),
        (Path('data/sft-train.jsonl'), 'data/sft-train.jsonl'),
        (Path('data/sft-validation.jsonl'), 'data/sft-validation.jsonl')
    ]
bundle = Path('dream-training-artifacts.zip')
with ZipFile(bundle, 'w', compression=ZIP_DEFLATED) as archive:
    for path, archive_name in artifact_entries:
        if not path.is_file():
            raise FileNotFoundError(f'Missing required training artifact: {path}')
        archive.write(path, arcname=archive_name)
    checksum_lines = []
    for path, archive_name in artifact_entries:
        digest = sha256(path.read_bytes()).hexdigest()
        checksum_lines.append(f'{digest}  {archive_name}')
    archive.writestr('SHA256SUMS.txt', '\n'.join(checksum_lines) + '\n')
print('Bundle size (bytes):', bundle.stat().st_size)
files.download(str(bundle))